# Overture — per-row licensing (live)

Overture aggregates many sources with different licenses. The backend surfaces a per-row `license_id` and warns when share-alike `ODbL-1.0` (OpenStreetMap-derived) rows are present — critical for commercial redistribution.

## Setup

Pull in the libraries used throughout: `EarthLens` is the unified entry point, `LicenseWarning` and `row_license` come from the Overture backend, and `geopandas` reads the written `parquet`.

In [ ]:
import warnings

import geopandas as gpd

from earthlens.earthlens import EarthLens
from earthlens.overture import LicenseWarning, row_license

## 1 · Define the area of interest

A single Times Square block, given as south/north latitudes and west/east longitudes, plus the output directory the buildings `parquet` will be written to.

In [ ]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

## 2 · Download the buildings layer

Build the Overture request first, then call `download()` on it. We run the download inside a `warnings.catch_warnings(record=True)` block so any `LicenseWarning` is captured rather than printed inline, letting us inspect it afterwards.

In [ ]:
overture = EarthLens(
    data_source="overture",
    dataset="buildings",
    variables=[],
    aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]],
    path=OUT,
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    paths = overture.download()

### License warnings

Filter the captured warnings down to `LicenseWarning` instances and print them — this is how the backend flags the presence of share-alike `ODbL-1.0` rows.

In [ ]:
for w in caught:
    if issubclass(w.category, LicenseWarning):
        print("LicenseWarning:", w.message)

## 3 · Per-row `license_id`

Read the written `parquet` back into a `GeoDataFrame` and tally the `license_id` column — every feature carries its own license, so the distribution is visible at a glance.

In [ ]:
gdf = gpd.read_parquet(paths[0])
gdf["license_id"].value_counts()

## 4 · The derivation rule

The rule behind `license_id` is exposed directly as `row_license` for ad-hoc analysis of a `sources` cell — `ODbL-1.0` wins whenever an OpenStreetMap source is present, even if it is not listed first.

In [ ]:
print(
    row_license(
        [
            {"dataset": "Overture", "license": "CDLA-Permissive-2.0"},
            {"dataset": "OpenStreetMap", "license": "ODbL-1.0"},
        ]
    )
)
print(row_license([{"dataset": "Foursquare", "license": "Apache-2.0"}]))